Data från: "https://www.kaggle.com/datasets/ulrikthygepedersen/online-retail-dataset"

In [1]:
# Importerar dependencies som behövs för det vi ska göra.
import pandas as pd
import duckdb
import plotly.express as px

# Connectar till databas som skapas och sedan fylls på i queryn nedan.
db = duckdb.connect("../../data/retail.db")
db.sql("SELECT current_database()")

┌────────────────────┐
│ current_database() │
│      varchar       │
├────────────────────┤
│ retail             │
└────────────────────┘

Skapar schema för att strukturera data-pipelinen. Ger även bättre struktur i arbetet senare med analys.

In [2]:
db.sql("CREATE SCHEMA IF NOT EXISTS raw;")
db.sql("CREATE SCHEMA IF NOT EXISTS staging;")
db.sql("CREATE SCHEMA IF NOT EXISTS mart;")

Laddar in data från csv filen till en tabell i duckdb med specifikt namn.
Gör det lättare att referera till datan i queries.


~~db.sql("""
       Create table raw.retail AS
       SELECT * FROM 'C:/Users/Decss/Desktop/retail-transaction-analyses/Data/online_retail.csv'
       """):~~

 
Ovan query arkiveras med fördel query i nedan kodblock. Nedan query laddar csv och sparar den i databasen vi specificerar i första cellen. Istället för temporärt minne så skapar vi en databas.

In [3]:
# Skapar raw.retail som första tabell.
db.sql("""
CREATE OR REPLACE TABLE raw.retail AS
SELECT *
FROM read_csv_auto('C:/Users/Decss/Desktop/retail-transaction-analyses/Data/online_retail.csv')
""")

In [4]:
# Kikar på hur tabellen ser ut. Bättre att använda pandas (se längre ned).
db.sql("describe raw.retail")

┌─────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│ column_name │ column_type │  null   │   key   │ default │  extra  │
│   varchar   │   varchar   │ varchar │ varchar │ varchar │ varchar │
├─────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ InvoiceNo   │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ StockCode   │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ Description │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ Quantity    │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ InvoiceDate │ TIMESTAMP   │ YES     │ NULL    │ NULL    │ NULL    │
│ UnitPrice   │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ CustomerID  │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ Country     │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
└─────────────┴─────────────┴─────────┴─────────┴─────────┴─────────┘

In [5]:
db.sql("SELECT * FROM raw.retail LIMIT 5")

┌───────────┬───────────┬─────────────────────────────────────┬──────────┬─────────────────────┬───────────┬────────────┬────────────────┐
│ InvoiceNo │ StockCode │             Description             │ Quantity │     InvoiceDate     │ UnitPrice │ CustomerID │    Country     │
│  varchar  │  varchar  │               varchar               │  int64   │      timestamp      │  double   │   double   │    varchar     │
├───────────┼───────────┼─────────────────────────────────────┼──────────┼─────────────────────┼───────────┼────────────┼────────────────┤
│ 536365    │ 85123A    │ WHITE HANGING HEART T-LIGHT HOLDER  │        6 │ 2010-12-01 08:26:00 │      2.55 │    17850.0 │ United Kingdom │
│ 536365    │ 71053     │ WHITE METAL LANTERN                 │        6 │ 2010-12-01 08:26:00 │      3.39 │    17850.0 │ United Kingdom │
│ 536365    │ 84406B    │ CREAM CUPID HEARTS COAT HANGER      │        8 │ 2010-12-01 08:26:00 │      2.75 │    17850.0 │ United Kingdom │
│ 536365    │ 84029G    │ K

In [6]:
# Räknar antalet rader i tabellen.
db.sql("SELECT COUNT(*) FROM raw.retail")


┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│       541909 │
└──────────────┘

In [7]:
# Kikar på data i invoiceno och customerid för att se hur mång aunika värden som finns.
db.sql("SELECT COUNT(DISTINCT invoiceno) , COUNT(DISTINCT CustomerID)FROM raw.retail")

┌───────────────────────────┬────────────────────────────┐
│ count(DISTINCT invoiceno) │ count(DISTINCT CustomerID) │
│           int64           │           int64            │
├───────────────────────────┼────────────────────────────┤
│                     25900 │                       4372 │
└───────────────────────────┴────────────────────────────┘

In [8]:
# Kollar efter potentiellt "orimliga" värden i quantity och unitprice.
# Har vi negativa värden eller 0 så finn det förmodligen en förklaring (alt något är fel).
db.sql("SELECT invoiceno, quantity, unitprice FROM raw.retail WHERE quantity < 0 OR unitprice <= 0 ")

┌───────────┬──────────┬───────────┐
│ InvoiceNo │ Quantity │ UnitPrice │
│  varchar  │  int64   │  double   │
├───────────┼──────────┼───────────┤
│ C536379   │       -1 │      27.5 │
│ C536383   │       -1 │      4.65 │
│ C536391   │      -12 │      1.65 │
│ C536391   │      -24 │      0.29 │
│ C536391   │      -24 │      0.29 │
│ C536391   │      -24 │      0.29 │
│ C536391   │      -12 │      3.45 │
│ C536391   │      -12 │      1.65 │
│ C536391   │      -24 │      1.65 │
│ 536414    │       56 │       0.0 │
│   ·       │        · │        ·  │
│   ·       │        · │        ·  │
│   ·       │        · │        ·  │
│ C573260   │       -1 │      1.45 │
│ C573268   │       -1 │     907.0 │
│ C573271   │       -2 │      1.95 │
│ C573283   │       -1 │      9.95 │
│ 573293    │       10 │       0.0 │
│ C573306   │       -1 │      9.95 │
│ C573306   │       -1 │      6.95 │
│ 573317    │      -84 │       0.0 │
│ 573318    │      -47 │       0.0 │
│ C573321   │       -1 │       8.5 │
├

In [9]:
# Använder pandas info,head,describe för att få en helhetbild av datan.
df = db.sql("SELECT * FROM raw.retail LIMIT 50000").df()
df.info()
df.head()
df.describe()


<class 'pandas.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 8 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   InvoiceNo    50000 non-null  str           
 1   StockCode    50000 non-null  str           
 2   Description  49858 non-null  str           
 3   Quantity     50000 non-null  int64         
 4   InvoiceDate  50000 non-null  datetime64[us]
 5   UnitPrice    50000 non-null  float64       
 6   CustomerID   32119 non-null  float64       
 7   Country      50000 non-null  str           
dtypes: datetime64[us](1), float64(2), int64(1), str(4)
memory usage: 3.1 MB


,Quantity,InvoiceDate,UnitPrice,CustomerID
count,50000.000000,50000,50000.000000,32119.000000
mean,8.293320,2010-12-14 18:35:52.846800,6.418361,15431.221987
min,-9360.000000,2010-12-01 08:26:00,0.000000,12347.000000
25%,1.000000,2010-12-06 13:13:00,1.250000,14057.000000
50%,2.000000,2010-12-10 15:17:00,2.510000,15464.000000
75%,8.000000,2010-12-19 13:59:00,4.250000,17059.000000
max,2880.000000,2011-01-09 15:13:00,16888.020000,18283.000000
std,55.528494,NaN,166.504827,1753.358819


In [10]:
# Kikar på data i kolumnerna för att bekanta mig med värdena i tabellen.
db.sql("SELECT DISTINCT invoiceno, stockcode, description, quantity, unitprice, invoicedate, customerid, country FROM raw.retail LIMIT 15")

┌───────────┬───────────┬────────────────────────────────────┬──────────┬───────────┬─────────────────────┬────────────┬────────────────┐
│ InvoiceNo │ StockCode │            Description             │ Quantity │ UnitPrice │     InvoiceDate     │ CustomerID │    Country     │
│  varchar  │  varchar  │              varchar               │  int64   │  double   │      timestamp      │   double   │    varchar     │
├───────────┼───────────┼────────────────────────────────────┼──────────┼───────────┼─────────────────────┼────────────┼────────────────┤
│ 578068    │ 21914     │ BLUE HARMONICA IN BOX              │        4 │      1.25 │ 2011-11-22 15:45:00 │    14505.0 │ United Kingdom │
│ 578068    │ 22383     │ LUNCH BAG SUKI DESIGN              │        2 │      1.65 │ 2011-11-22 15:45:00 │    14505.0 │ United Kingdom │
│ 578072    │ 21084     │ SET/6 COLLAGE PAPER CUPS           │        3 │      0.19 │ 2011-11-22 16:02:00 │    17759.0 │ United Kingdom │
│ 578072    │ 15034     │ PAPER PO

In [11]:
# Finns null i customerid?
db.sql("SELECT customerid FROM raw.retail where customerid is null limit 5")

┌────────────┐
│ CustomerID │
│   double   │
├────────────┤
│       NULL │
│       NULL │
│       NULL │
│       NULL │
│       NULL │
└────────────┘

In [12]:
# Finns null i description?
db.sql("SELECT description from raw.retail where description is null limit 5")

┌─────────────┐
│ Description │
│   varchar   │
├─────────────┤
│ NULL        │
│ NULL        │
│ NULL        │
│ NULL        │
│ NULL        │
└─────────────┘

In [13]:
# Kollar efter dubbletter i datan genom att räkna totala antal rader och specia samt räkna unika rader.
db.sql("SELECT count(*) as total_rows, count(distinct (invoiceno, stockcode, description, quantity, unitprice, invoicedate, customerid, country)) as distinct_rows FROM raw.retail")
# Vi kan se att det finns 541909  mot 536641 unika. Det finns alltså ett antal dubbletter i datan.

┌────────────┬───────────────┐
│ total_rows │ distinct_rows │
│   int64    │     int64     │
├────────────┼───────────────┤
│     541909 │        536641 │
└────────────┴───────────────┘

In [14]:
# För att ta reda på vilka specifika rader som är dubbletter använder vi följande query.
db.sql("SELECT invoiceno, stockcode, description, quantity, unitprice, invoicedate, customerid, country, COUNT(*) as cnt FROM raw.retail GROUP BY ALL HAVING COUNT(*) > 1")
# Nu har vi fått fram alla rader som har dubbletter (en eller flera). Iom skillnaden 5 268 mot 4970 här så 
# är en del av dubbletterna tripletter eller mer. 
# Innan vi börjar ta bort något så gör vi några fler kontroller, men adderar 
# dubbletter på listan över sådans som ska tas bort.


┌───────────┬───────────┬─────────────────────────────────────┬──────────┬───────────┬─────────────────────┬────────────┬────────────────┬───────┐
│ InvoiceNo │ StockCode │             Description             │ Quantity │ UnitPrice │     InvoiceDate     │ CustomerID │    Country     │  cnt  │
│  varchar  │  varchar  │               varchar               │  int64   │  double   │      timestamp      │   double   │    varchar     │ int64 │
├───────────┼───────────┼─────────────────────────────────────┼──────────┼───────────┼─────────────────────┼────────────┼────────────────┼───────┤
│ 578068    │ 23138     │ SINGLE WIRE HOOK IVORY HEART        │        2 │      1.45 │ 2011-11-22 15:45:00 │    14505.0 │ United Kingdom │     2 │
│ 578521    │ 22910     │ PAPER CHAIN KIT VINTAGE CHRISTMAS   │        1 │      2.95 │ 2011-11-24 14:06:00 │    17812.0 │ United Kingdom │     2 │
│ 578620    │ 22867     │ HAND WARMER BIRD DESIGN             │        1 │       2.1 │ 2011-11-24 15:21:00 │    17247.

In [15]:
# Kollar om vi har fall där unitprice = 0.
db.sql("SELECT invoiceno, stockcode, unitprice FROM raw.retail WHERE unitprice = 0 ")

┌───────────┬───────────┬───────────┐
│ InvoiceNo │ StockCode │ UnitPrice │
│  varchar  │  varchar  │  double   │
├───────────┼───────────┼───────────┤
│ 536414    │ 22139     │       0.0 │
│ 536545    │ 21134     │       0.0 │
│ 536546    │ 22145     │       0.0 │
│ 536547    │ 37509     │       0.0 │
│ 536549    │ 85226A    │       0.0 │
│ 536550    │ 85044     │       0.0 │
│ 536552    │ 20950     │       0.0 │
│ 536553    │ 37461     │       0.0 │
│ 536554    │ 84670     │       0.0 │
│ 536589    │ 21777     │       0.0 │
│   ·       │   ·       │        ·  │
│   ·       │   ·       │        ·  │
│   ·       │   ·       │        ·  │
│ 581210    │ 23395     │       0.0 │
│ 581211    │ 22142     │       0.0 │
│ 581212    │ 22578     │       0.0 │
│ 581213    │ 22576     │       0.0 │
│ 581226    │ 23090     │       0.0 │
│ 581234    │ 72817     │       0.0 │
│ 581406    │ 46000M    │       0.0 │
│ 581406    │ 46000S    │       0.0 │
│ 581408    │ 85175     │       0.0 │
│ 581422    

In [16]:
# Kollar efter rader där antingen null eller blankt i invoiceno (även stockcode).
# Detta eftersom de två kolumnerna är viktiga för att identifiera transaktioner och produkter.
# Utan dessa är datan nästan oanvändbar och vi bör då överväga att ta bort dessa rader.
db.sql("SELECT invoiceno FROM raw.retail WHERE invoiceno = ' '")

┌───────────┐
│ InvoiceNo │
│  varchar  │
├───────────┤
│  0 rows   │
└───────────┘

In [17]:
# Vi tar bort dubletter i datan. Detta görs genom att skapa en ny tabell och välja alla unika rader från alla kolumner i den gamla tabellen.
db.sql("""
    CREATE OR REPLACE TABLE staging.retail_clean AS 
    SELECT DISTINCT * 
    FROM raw.retail
""")

In [18]:
# Ovan teknik ska ha tagit bort alla dubbletter. Men vi kikar så att det stämmer och ser rimligt ut.
db.sql("SELECT count(*) as total_rows, count(distinct (invoiceno, stockcode, description, quantity, unitprice, invoicedate, customerid, country)) as distinct_rows FROM staging.retail_clean")

┌────────────┬───────────────┐
│ total_rows │ distinct_rows │
│   int64    │     int64     │
├────────────┼───────────────┤
│     536641 │        536641 │
└────────────┴───────────────┘

In [19]:
# Vi ser även till att trimma bort eventuella blanksteg i kolumnen stockcode och invoiceno.
db.sql("""
       SELECT 
       invoiceno, 
       stockcode 
       FROM staging.retail_clean 
       WHERE stockcode LIKE '% %'""")

┌───────────┬──────────────┐
│ InvoiceNo │  StockCode   │
│  varchar  │   varchar    │
├───────────┼──────────────┤
│ C550932   │ BANK CHARGES │
│ C570463   │ BANK CHARGES │
│ 568375    │ BANK CHARGES │
│ 551945    │ BANK CHARGES │
│ C567136   │ BANK CHARGES │
│ C558349   │ BANK CHARGES │
│ 536779    │ BANK CHARGES │
│ C563535   │ BANK CHARGES │
│ 581127    │ BANK CHARGES │
│ 579137    │ BANK CHARGES │
│   ·       │      ·       │
│   ·       │      ·       │
│   ·       │      ·       │
│ C560823   │ BANK CHARGES │
│ C577342   │ BANK CHARGES │
│ C544575   │ BANK CHARGES │
│ C537572   │ BANK CHARGES │
│ C546943   │ BANK CHARGES │
│ C553511   │ BANK CHARGES │
│ C577343   │ BANK CHARGES │
│ C564498   │ BANK CHARGES │
│ C572346   │ BANK CHARGES │
│ 565735    │ BANK CHARGES │
├───────────┴──────────────┤
│ 37 rows        2 columns │
│ (20 shown)               │
└──────────────────────────┘

In [20]:
# Ser till att namn är enhetliga och att vi rensar evetuella blanksteg som är fel.
db.sql("""
       CREATE OR REPLACE TABLE staging.retail_clean AS
       SELECT 
       * EXCLUDE (stockcode, invoiceno),
       TRIM(UPPER(stockcode)) as stockcode,
       TRIM(UPPER(invoiceno)) as invoiceno
       FROM staging.retail_clean
       """)

In [21]:
# Vi kikar på quantity kolumnen för att kunna avgöra vad som är extrema värden.
# Används senare vid flaggning. 
# Här ser vi att det börjar dra iväg efter 95e percentilen.
# För att dessa inte ska påverka beräkningar, och analyser generellt, så flaggar vi dessa som extrema värden.
# Det blir då lättare att inkludera eller exkludera denna data.
df_quantity = db.sql("SELECT Quantity FROM staging.retail_clean").df()
df_quantity["Quantity"].quantile([0.95, 0.975, 0.99, 0.999])


0.950     30.0
0.975     48.0
0.990    100.0
0.999    480.0
Name: Quantity, dtype: float64

In [22]:
# Samma sak här. Kikar på hur UnitPrice beter sig. Vilken nivå vi ska lägga vår flaggning på.
# Vi kan se att pris börjar dra iväg efter 99e percentilen. (till skillnad från 95e percentilen i quantity).
df_unitprice = db.sql("SELECT UnitPrice FROM staging.retail_clean").df()
df_unitprice["UnitPrice"].quantile([0.95, 0.975, 0.99, 0.999])


0.950      9.9500
0.975     12.7500
0.990     18.0000
0.999    207.9088
Name: UnitPrice, dtype: float64

In [23]:
# Inför flaggning så behöver vi kika på country och se vad vi har för olika värden där.
df_country = db.sql("SELECT DISTINCT country FROM staging.retail_clean ORDER BY country asc").df()
print(df_country)

                 Country
0              Australia
1                Austria
2                Bahrain
3                Belgium
4                 Brazil
5                 Canada
6        Channel Islands
7                 Cyprus
8         Czech Republic
9                Denmark
10                  EIRE
11    European Community
12               Finland
13                France
14               Germany
15                Greece
16             Hong Kong
17               Iceland
18                Israel
19                 Italy
20                 Japan
21               Lebanon
22             Lithuania
23                 Malta
24           Netherlands
25                Norway
26                Poland
27              Portugal
28                   RSA
29          Saudi Arabia
30             Singapore
31                 Spain
32                Sweden
33           Switzerland
34                   USA
35  United Arab Emirates
36        United Kingdom
37           Unspecified


In [24]:
country = df_country["Country"].tolist()
print(country)

['Australia', 'Austria', 'Bahrain', 'Belgium', 'Brazil', 'Canada', 'Channel Islands', 'Cyprus', 'Czech Republic', 'Denmark', 'EIRE', 'European Community', 'Finland', 'France', 'Germany', 'Greece', 'Hong Kong', 'Iceland', 'Israel', 'Italy', 'Japan', 'Lebanon', 'Lithuania', 'Malta', 'Netherlands', 'Norway', 'Poland', 'Portugal', 'RSA', 'Saudi Arabia', 'Singapore', 'Spain', 'Sweden', 'Switzerland', 'USA', 'United Arab Emirates', 'United Kingdom', 'Unspecified']


In [25]:
# Efter att vi har rensat dubbletter så är det dags att addera de identifierade flaggor/klassificeringarna till tabellen.
# Vi gick igenom varje kolumn var för sig och kom fram till flaggor i nedan query.
# Dessa skapas i staging.retail_clean som är den rensade och flaggade versionen av raw.retail
db.sql("""CREATE OR REPLACE TABLE staging.retail_clean AS
       SELECT *,
       CASE WHEN invoiceno like 'C%' THEN 1 ELSE 0 END as is_credit_invoice,
       CASE 
       WHEN stockcode ILIKE 'M' THEN 0
       WHEN stockcode ILIKE 'DOT' THEN 0
       WHEN stockcode ILIKE 'POST' THEN 0
       WHEN description ILIKE '%AMAZON%' THEN 0
       WHEN description ILIKE '%MANUAL%' THEN 0
       WHEN description ILIKE '%POST%' THEN 0
       ELSE 1
       END AS is_product,

       CASE
       WHEN unitprice >= 100 then 'high_end_price'
       WHEN unitprice >= 20 then 'premium_price'
       ELSE 'normal_price' END as price_segment,

       CASE WHEN description is null OR TRIM(description) = ' ' THEN 1 ELSE 0 END as is_missing_description,
       CASE WHEN quantity < 0 THEN 1 ELSE 0 END as is_negative_quantity,
       CASE WHEN quantity > 50 THEN 1 ELSE 0 END as is_extreme_quantity,
       CASE WHEN quantity = 0 THEN 1 ELSE 0 END as is_zero_quantity,
       CASE WHEN unitprice < 0 THEN 1 ELSE 0 END as is_negative_unitprice,
       CASE WHEN unitprice > 20 THEN 1 ELSE 0 END as is_extreme_unitprice,
       CASE WHEN unitprice = 0 THEN 1 ELSE 0 END as is_zero_unitprice,
       CASE WHEN customerid is null THEN 1 ELSE 0 END as is_missing_customerid,
       CASE WHEN stockcode is null OR stockcode = ' ' THEN 1 ELSE 0 END as is_missing_stockcode,
       CASE WHEN invoiceno is null OR invoiceno = ' ' THEN 1 ELSE 0 END as is_missing_invoiceno,
       CASE WHEN country NOT IN ('Australia', 'Austria', 'Bahrain', 'Belgium', 'Brazil', 'Canada', 'Cyprus', 'Czech Republic', 'Denmark','Finland', 'France', 'Germany', 'Greece', 'Hong Kong', 'Iceland', 'Israel', 'Italy', 'Japan', 'Lebanon', 'Lithuania', 'Malta', 'Netherlands', 'Norway', 'Poland', 'Portugal', 'RSA', 'Saudi Arabia', 'Singapore', 'Spain', 'Sweden', 'Switzerland', 'USA', 'United Arab Emirates', 'United Kingdom') THEN 1 ELSE 0 END as is_unusual_country
       FROM staging.retail_clean
       """)
# Vi kontrollerade saknad InvoiceNo och StockCode. Inga sådana rader fanns, därför är flaggorna alltid 0.

In [26]:
# Kikar på den nya tabellen och stämmer av flaggorna.
db.sql("SELECT COUNT(*) FROM staging.retail_clean WHERE is_missing_customerid = 1")


┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│       135037 │
└──────────────┘

In [27]:
# Att kika på samförhållande mellan flaggor, exempelvis kreditfakturor och negativa värden, ger oss en
# bättre bild av hur verksamheten fungerar.
db.sql("SELECT * FROM staging.retail_clean WHERE is_credit_invoice = 1 AND is_negative_quantity = 1")

┌────────────────────────────────────┬──────────┬─────────────────────┬───────────┬────────────┬────────────────┬───────────┬───────────┬───────────────────┬────────────┬───────────────┬────────────────────────┬──────────────────────┬─────────────────────┬──────────────────┬───────────────────────┬──────────────────────┬───────────────────┬───────────────────────┬──────────────────────┬──────────────────────┬────────────────────┐
│            Description             │ Quantity │     InvoiceDate     │ UnitPrice │ CustomerID │    Country     │ stockcode │ invoiceno │ is_credit_invoice │ is_product │ price_segment │ is_missing_description │ is_negative_quantity │ is_extreme_quantity │ is_zero_quantity │ is_negative_unitprice │ is_extreme_unitprice │ is_zero_unitprice │ is_missing_customerid │ is_missing_stockcode │ is_missing_invoiceno │ is_unusual_country │
│              varchar               │  int64   │      timestamp      │  double   │   double   │    varchar     │  varchar  │  varch

In [28]:
# Skapar en tabell och summerar antal rader som har flaggats i de olika kategorierna.
db.sql("""
SELECT 
       SUM(is_credit_invoice) as credit_invoices,
       SUM(is_missing_description) as missing_descriptions,
       SUM(is_negative_quantity) as negative_quantities,
       SUM(is_extreme_quantity) as extreme_quantities,
       SUM(is_zero_quantity) as zero_quantities,
       SUM(is_negative_unitprice) as negative_unitprices,
       SUM(is_extreme_unitprice) as extreme_unitprices,
       SUM(is_zero_unitprice) as zero_unitprices,
       SUM(is_missing_customerid) as missing_customerids,
       SUM(is_missing_stockcode) as missing_stockcodes,
       SUM(is_missing_invoiceno) as missing_invoicenumbers,
       SUM(is_unusual_country) as unusual_countries
FROM staging.retail_clean
""")

┌─────────────────┬──────────────────────┬─────────────────────┬────────────────────┬─────────────────┬─────────────────────┬────────────────────┬─────────────────┬─────────────────────┬────────────────────┬────────────────────────┬───────────────────┐
│ credit_invoices │ missing_descriptions │ negative_quantities │ extreme_quantities │ zero_quantities │ negative_unitprices │ extreme_unitprices │ zero_unitprices │ missing_customerids │ missing_stockcodes │ missing_invoicenumbers │ unusual_countries │
│     int128      │        int128        │       int128        │       int128       │     int128      │       int128        │       int128       │     int128      │       int128        │       int128       │         int128         │      int128       │
├─────────────────┼──────────────────────┼─────────────────────┼────────────────────┼─────────────────┼─────────────────────┼────────────────────┼─────────────────┼─────────────────────┼────────────────────┼────────────────────────┼─────────

In [29]:
# Efter en överblick av antal flaggor så tar vi och kikar på vad för extrema värden vi har i de olika kategorierna.
# Vi börjar med att kika närmare på de numeriska kolumnerna och vad för typ av "extrema" värden vi hittar där.
db.sql("SELECT invoiceno, stockcode, description, quantity, unitprice FROM staging.retail_clean ORDER BY quantity DESC LIMIT 20")

┌───────────┬───────────┬─────────────────────────────────────┬──────────┬───────────┐
│ invoiceno │ stockcode │             Description             │ Quantity │ UnitPrice │
│  varchar  │  varchar  │               varchar               │  int64   │  double   │
├───────────┼───────────┼─────────────────────────────────────┼──────────┼───────────┤
│ 581483    │ 23843     │ PAPER CRAFT , LITTLE BIRDIE         │    80995 │      2.08 │
│ 541431    │ 23166     │ MEDIUM CERAMIC TOP STORAGE JAR      │    74215 │      1.04 │
│ 578841    │ 84826     │ ASSTD DESIGN 3D PAPER STICKERS      │    12540 │       0.0 │
│ 542504    │ 37413     │ NULL                                │     5568 │       0.0 │
│ 573008    │ 84077     │ WORLD WAR 2 GLIDERS ASSTD DESIGNS   │     4800 │      0.21 │
│ 554868    │ 22197     │ SMALL POPCORN HOLDER                │     4300 │      0.72 │
│ 556231    │ 85123A    │ ?                                   │     4000 │       0.0 │
│ 544612    │ 22053     │ EMPIRE DESIGN ROS

In [30]:
# Skapar en dataframe för att sammanställa alla rader med anomalier. Just nu bara quantity (q).
# När vi har fler kategorier som denna så slår vi till sist ihop dem till en enskild tabell 
# där alla anomalier finns samlade.
top_anomalies_q = db.sql("""
    SELECT *, 'extreme_quantity' AS anomaly_type
    FROM staging.retail_clean 
    WHERE quantity >= 74215
""")
# Skapar en dataframe för unitprice där vi gör samma sak.Vi kikar först på de outliers som finns i kolumnen.
top_anomalies_p = db.sql("""
    SELECT *, 'extreme_unitprice' AS anomaly_type
    FROM staging.retail_clean 
    WHERE unitprice >= 500
""")

# Skapar dataframe för customerid vars beteende sticker ut avsevärt.
top_anomalies_c = db.sql("""
    SELECT s.*, 'anomaly_customer' AS anomaly_type
    FROM staging.retail_clean s
    WHERE s.customerid IN (
        SELECT customerid
        FROM staging.retail_clean
        GROUP BY customerid
        HAVING SUM(CASE WHEN quantity * unitprice > 0 THEN quantity * unitprice END) > 10000
        AND ABS(SUM(quantity * unitprice)) < 
            SUM(CASE WHEN quantity * unitprice > 0 THEN quantity * unitprice END) * 0.05
    )
""")

In [31]:
# Verifierar kolumnantal innan UNION ALL
print(len(top_anomalies_q.columns))
print(len(top_anomalies_p.columns))
print(len(top_anomalies_c.columns))

23
23
23


In [32]:
# Slår ihop samtliga anomalitabeller till staging.top_anomalies.
db.sql("""
    CREATE OR REPLACE TABLE staging.top_anomalies AS
    SELECT * FROM top_anomalies_q
    UNION ALL
    SELECT * FROM top_anomalies_p
    UNION ALL
    SELECT * FROM top_anomalies_c
""")

In [33]:
db.sql("SELECT * FROM staging.retail_clean ORDER BY unitprice DESC limit 50")

┌─────────────────┬──────────┬─────────────────────┬───────────┬────────────┬────────────────┬───────────┬───────────┬───────────────────┬────────────┬────────────────┬────────────────────────┬──────────────────────┬─────────────────────┬──────────────────┬───────────────────────┬──────────────────────┬───────────────────┬───────────────────────┬──────────────────────┬──────────────────────┬────────────────────┐
│   Description   │ Quantity │     InvoiceDate     │ UnitPrice │ CustomerID │    Country     │ stockcode │ invoiceno │ is_credit_invoice │ is_product │ price_segment  │ is_missing_description │ is_negative_quantity │ is_extreme_quantity │ is_zero_quantity │ is_negative_unitprice │ is_extreme_unitprice │ is_zero_unitprice │ is_missing_customerid │ is_missing_stockcode │ is_missing_invoiceno │ is_unusual_country │
│     varchar     │  int64   │      timestamp      │  double   │   double   │    varchar     │  varchar  │  varchar  │       int32       │   int32    │    varchar     │

In [34]:
# Kikar efter extrema prisvärden. Vi har redan flaggat alla över pris 20, men kikar nu lite närmare på vad som är
# extrema värden inom flaggan. För att kunna se extrema priser på faktiska produkter och inte affärslogik
# så lägger vi till några filter.
db.sql("SELECT * FROM staging.retail_clean WHERE invoiceno not like 'C%' AND invoiceno not ilike 'A%' AND description not ilike '%Amazon%' AND description not ilike '%Post%' AND description not ilike '%manual%' ORDER BY unitprice DESC limit 25 ")
# Resultatet visar att en del prdukter har ganska höga priser, men det är få. Sett till vad det 
# är för produkter så är priserna inte orimliga.

┌────────────────────────────────────┬──────────┬─────────────────────┬───────────┬────────────┬────────────────┬───────────┬───────────┬───────────────────┬────────────┬────────────────┬────────────────────────┬──────────────────────┬─────────────────────┬──────────────────┬───────────────────────┬──────────────────────┬───────────────────┬───────────────────────┬──────────────────────┬──────────────────────┬────────────────────┐
│            Description             │ Quantity │     InvoiceDate     │ UnitPrice │ CustomerID │    Country     │ stockcode │ invoiceno │ is_credit_invoice │ is_product │ price_segment  │ is_missing_description │ is_negative_quantity │ is_extreme_quantity │ is_zero_quantity │ is_negative_unitprice │ is_extreme_unitprice │ is_zero_unitprice │ is_missing_customerid │ is_missing_stockcode │ is_missing_invoiceno │ is_unusual_country │
│              varchar               │  int64   │      timestamp      │  double   │   double   │    varchar     │  varchar  │  var

I och med att vi identifierat att det existerar en del extrema värden som är långt över de 20 där vår cutoff var så kan vi överväga att antingen lägga till fler flaggor eller att segmentera dessa. 

In [35]:
# Vi går vidare (färdiga kolumner: invoiceno, stockcode, unitprice, quantity, description)
# till invoicedate, customerid, country.
db.sql("SELECT country, count(country) FROM staging.retail_clean GROUP BY country ORDER BY count(country) desc")
# Det enda vi behövde göra i Country vad i princip att flagga de udda landskategorierna. Därefter så är resterande
# affärslogik. Inget mer behöver göras i Country.

┌──────────────────────┬────────────────┐
│       Country        │ count(country) │
│       varchar        │     int64      │
├──────────────────────┼────────────────┤
│ United Kingdom       │         490300 │
│ Germany              │           9480 │
│ France               │           8541 │
│ EIRE                 │           8184 │
│ Spain                │           2528 │
│ Netherlands          │           2371 │
│ Belgium              │           2069 │
│ Switzerland          │           1994 │
│ Portugal             │           1510 │
│ Australia            │           1258 │
│   ·                  │             ·  │
│   ·                  │             ·  │
│   ·                  │             ·  │
│ Malta                │            127 │
│ United Arab Emirates │             68 │
│ European Community   │             61 │
│ RSA                  │             58 │
│ Lebanon              │             45 │
│ Lithuania            │             35 │
│ Brazil               │          

In [36]:
# Vi går vidare till CustomerID och bekantar oss med datan samt kikar om något mer än den flagga vi la till förut behöver ordnas.
db.sql("SELECT customerid, count(customerid) FROM staging.retail_clean GROUP BY customerid ORDER BY count(customerid) ASC")

┌────────────┬───────────────────┐
│ CustomerID │ count(customerid) │
│   double   │       int64       │
├────────────┼───────────────────┤
│       NULL │                 0 │
│    13307.0 │                 1 │
│    15753.0 │                 1 │
│    13256.0 │                 1 │
│    13017.0 │                 1 │
│    16579.0 │                 1 │
│    16738.0 │                 1 │
│    18174.0 │                 1 │
│    16990.0 │                 1 │
│    13391.0 │                 1 │
│       ·    │                 · │
│       ·    │                 · │
│       ·    │                 · │
│    14298.0 │              1640 │
│    13263.0 │              1667 │
│    13089.0 │              1853 │
│    14646.0 │              2085 │
│    15311.0 │              2478 │
│    14606.0 │              2759 │
│    12748.0 │              4459 │
│    14096.0 │              5128 │
│    14911.0 │              5898 │
│    17841.0 │              7812 │
├────────────┴───────────────────┤
│ 4373 rows (20 show

In [37]:
# Här kan vi dubbelkolla så att det inte finns några konstiga tecken i customerid.
db.sql("""SELECT customerid
FROM staging.retail_clean
WHERE customerid IS NOT NULL
  AND CAST(CAST(customerid AS INT) AS VARCHAR) !~ '^[0-9]+$'
""")
# 0 rader identifierades. Rimligt att anta att inget mer än flaggan för nulls behöver göras på denna kolumn för tillfället.
# Övriga segmenteringar får vänta tills enrichment/feature engineering.

┌────────────┐
│ CustomerID │
│   double   │
├────────────┤
│   0 rows   │
└────────────┘

In [38]:
# Kikar högst upp och längst ned för att stämma av att rangen ser ok ut. Inga datum utanför vad som är logiskt korrekt.
# Detta syns även i description? MIN och MAX
db.sql("SELECT invoicedate FROM staging.retail_clean ORDER BY invoicedate asc limit 25")

┌─────────────────────┐
│     InvoiceDate     │
│      timestamp      │
├─────────────────────┤
│ 2010-12-01 08:26:00 │
│ 2010-12-01 08:26:00 │
│ 2010-12-01 08:26:00 │
│ 2010-12-01 08:26:00 │
│ 2010-12-01 08:26:00 │
│ 2010-12-01 08:26:00 │
│ 2010-12-01 08:26:00 │
│ 2010-12-01 08:28:00 │
│ 2010-12-01 08:28:00 │
│ 2010-12-01 08:34:00 │
│          ·          │
│          ·          │
│          ·          │
│ 2010-12-01 08:34:00 │
│ 2010-12-01 08:34:00 │
│ 2010-12-01 08:34:00 │
│ 2010-12-01 08:34:00 │
│ 2010-12-01 08:34:00 │
│ 2010-12-01 08:34:00 │
│ 2010-12-01 08:34:00 │
│ 2010-12-01 08:34:00 │
│ 2010-12-01 08:34:00 │
│ 2010-12-01 08:34:00 │
├─────────────────────┤
│ 25 rows (20 shown)  │
└─────────────────────┘

In [39]:
# Kollar datatyper utifall det skulle finnas någon rad som inte är timestamp.
db.sql("SELECT typeof(invoicedate) FROM staging.retail_clean order by typeof(invoicedate) asc limit 10")
# Efter detta så har vi kikat på alla kolumner. Vi har flaggat det vi behöver så här långt. Och i 
# de fåtal fall där vi hittade extrema värden även i relation till vad som flaggats så har vi märkt
# dessa som top_anomalies.


┌─────────────────────┐
│ typeof(invoicedate) │
│       varchar       │
├─────────────────────┤
│ TIMESTAMP           │
│ TIMESTAMP           │
│ TIMESTAMP           │
│ TIMESTAMP           │
│ TIMESTAMP           │
│ TIMESTAMP           │
│ TIMESTAMP           │
│ TIMESTAMP           │
│ TIMESTAMP           │
│ TIMESTAMP           │
├─────────────────────┤
│       10 rows       │
└─────────────────────┘

In [40]:
# Sista kolla på min och max i quantity för att säkerställa att vi inte missade något extremt.
db.sql("SELECT * FROM staging.retail_clean ORDER BY quantity asc limit 20")

# Med detta gjort är det dags för feature-engineering/enrichment

┌─────────────────────────────────────┬──────────┬─────────────────────┬───────────┬────────────┬────────────────┬───────────┬───────────┬───────────────────┬────────────┬───────────────┬────────────────────────┬──────────────────────┬─────────────────────┬──────────────────┬───────────────────────┬──────────────────────┬───────────────────┬───────────────────────┬──────────────────────┬──────────────────────┬────────────────────┐
│             Description             │ Quantity │     InvoiceDate     │ UnitPrice │ CustomerID │    Country     │ stockcode │ invoiceno │ is_credit_invoice │ is_product │ price_segment │ is_missing_description │ is_negative_quantity │ is_extreme_quantity │ is_zero_quantity │ is_negative_unitprice │ is_extreme_unitprice │ is_zero_unitprice │ is_missing_customerid │ is_missing_stockcode │ is_missing_invoiceno │ is_unusual_country │
│               varchar               │  int64   │      timestamp      │  double   │   double   │    varchar     │  varchar  │  va

In [41]:
# Kollar antal fakturor/transaktioner uppdelat på månader för att använda som underlag för feature i mart.retail_enriched.
# "is_peak_season" blir då månad 11 och 12
db.sql("SELECT EXTRACT(month from invoicedate) as month, count(*) as transactions from staging.retail_clean GROUP BY month ORDER BY transactions DESC")

┌───────┬──────────────┐
│ month │ transactions │
│ int64 │    int64     │
├───────┼──────────────┤
│    11 │        83343 │
│    12 │        67227 │
│    10 │        59969 │
│     9 │        49861 │
│     7 │        39267 │
│     5 │        36782 │
│     6 │        36609 │
│     3 │        36439 │
│     8 │        35064 │
│     1 │        34900 │
│     4 │        29701 │
│     2 │        27479 │
├───────┴──────────────┤
│ 12 rows    2 columns │
└──────────────────────┘

In [42]:
# Vi ska ladda in datum för högtider så att vi kan specificera dem i nedan enrichment.
# Separat tidsdimension för högtider.
db.sql("""Create or Replace Table holidays AS
       SELECT * 
       FROM read_csv_auto('C:/Users/Decss/Desktop/retail-transaction-analyses/Data/holiday_dates.csv')
       """)

In [43]:
# Detta kan klassas som "Faktatabellen" i star-schema
# I steget där vi berikar datan kan vi börja med tidsaspekten. Segmentera invoicedate
# Viktigt är att vi "Förgrenar" tabellen här. staging.retail_clean får stanna som den är och vi skapar då
# En ny "mart.retail_enriched" där vi lägger till all berikning vi behöver.

# Tidsbaserad enrichment
db.sql("""create or replace table mart.retail_enriched AS
       SELECT *,
       EXTRACT(year from invoicedate) as year,
       EXTRACT(month from invoicedate) as month,
       EXTRACT(dow from invoicedate) as weekday_number,
       dayname(invoicedate) as weekday_name,
       EXTRACT(hour from invoicedate) as hour,
       CASE
            WHEN EXTRACT(hour from invoicedate) BETWEEN 0 AND 5 THEN 'Night'
            WHEN EXTRACT(hour from invoicedate) BETWEEN 6 AND 11 THEN 'Morning'
            WHEN EXTRACT(hour from invoicedate) BETWEEN 12 AND 17 THEN 'Afternoon'
            ELSE 'Evening'
       END AS time_of_day,
       EXTRACT(week from invoicedate) as week,
       CASE
            WHEN EXTRACT(month from invoicedate) IN (12, 1, 2) THEN 'Winter'
            WHEN EXTRACT(month from invoicedate) IN (3, 4, 5) THEN 'Spring'
            WHEN EXTRACT(month from invoicedate) IN (6, 7, 8) THEN 'Summer'
            ELSE 'Fall'
       END as Season,
       CASE
            WHEN EXTRACT(dow from invoicedate) IN (1, 2, 3, 4, 5) THEN 'Weekday'
            ELSE 'Weekend'
       END as day_type,
       CASE
            WHEN EXTRACT(month from invoicedate) IN (1, 2, 3) THEN 'Q1'
            WHEN EXTRACT(month from invoicedate) IN (4, 5, 6) THEN 'Q2'
            WHEN EXTRACT(month from invoicedate) IN (7, 8, 9) THEN 'Q3'
            ELSE 'Q4'
       END as quarter,
       CASE
            WHEN EXTRACT(month from invoicedate) IN (11, 12) THEN 1
            ELSE 0
       END as is_peak_season,

       CASE
            WHEN DATE(invoicedate) BETWEEN date_trunc('month', DATE(invoicedate)) + INTERVAL 24 DAY
            AND date_trunc('month', DATE(invoicedate)) + INTERVAl 28 DAY
            THEN 1 ELSE 0
       END as is_payday_window


       FROM staging.retail_clean
       """)
# payday_window skapas på så sätt att vi definierar ett intervall. date_trunc gör om till första i månaden. sedan plussas intervall på 24 och 28 dagar
# sedan så bestämmer man att payday är mellan dessa. infaller det mellan dessa är det 1 annars 0.

In [44]:
# Granskar resultatet efter att vi lagt till ett par features.
db.sql("SELECT * FROM mart.retail_enriched limit 10")

┌─────────────────────────────────┬──────────┬─────────────────────┬───────────┬────────────┬────────────────┬───────────┬───────────┬───────────────────┬────────────┬───────────────┬────────────────────────┬──────────────────────┬─────────────────────┬──────────────────┬───────────────────────┬──────────────────────┬───────────────────┬───────────────────────┬──────────────────────┬──────────────────────┬────────────────────┬───────┬───────┬────────────────┬──────────────┬───────┬─────────────┬───────┬─────────┬──────────┬─────────┬────────────────┬──────────────────┐
│           Description           │ Quantity │     InvoiceDate     │ UnitPrice │ CustomerID │    Country     │ stockcode │ invoiceno │ is_credit_invoice │ is_product │ price_segment │ is_missing_description │ is_negative_quantity │ is_extreme_quantity │ is_zero_quantity │ is_negative_unitprice │ is_extreme_unitprice │ is_zero_unitprice │ is_missing_customerid │ is_missing_stockcode │ is_missing_invoiceno │ is_unusual_cou

In [45]:
# Här "LEFT JOINar" vi tidsdimensionen holidays på mart.retail_enriched.
db.sql("""CREATE OR REPLACE TABLE mart.retail_enriched AS
       SELECT r.*,
       h.holiday_name,
       h.window_name as holiday_window,
       CASE WHEN h.holiday_name IS NOT NULL THEN 1 ELSE 0 END as is_holiday,
       CASE WHEN DATE(r.invoicedate) BETWEEN h.window_start AND h.window_end THEN 1 ELSE 0 END as is_holiday_window
       FROM mart.retail_enriched r
       LEFT JOIN holidays h 
       ON DATE(r.invoicedate) BETWEEN h.window_start AND h.window_end
       OR DATE(r.invoicedate) = h.holiday_date;

       """)

In [46]:
# Vi fortsätter vidare med features på radnivå.
# Uppdaterar "Faktatabellen" med "Transaktionsbaserad-enrichment"

db.sql("""CREATE OR REPLACE TABLE mart.retail_enriched AS
       SELECT r.*,

       quantity * unitprice as total_price,
       COUNT(*) OVER(PARTITION BY invoiceno) as basket_lines,
       SUM(quantity) OVER(PARTITION BY invoiceno) as basket_size,
       sum(quantity * unitprice) OVER(PARTITION BY invoiceno) as basket_value,
       COUNT(DISTINCT stockcode) OVER(PARTITION BY invoiceno) as basket_unique_item,
       CASE WHEN quantity > 0 THEN (quantity * unitprice) / NULLIF(SUM(CASE WHEN quantity > 0 THEN quantity * unitprice END) OVER (PARTITION BY invoiceno), 0) END as share_of_basket,
       CASE WHEN SUM(quantity) OVER(PARTITION BY invoiceno) >= 240 THEN 1 ELSE 0 END as is_large_basket,
       CASE WHEN sum(quantity * unitprice) OVER(PARTITION BY invoiceno) >= 400 THEN 1 ELSE 0 END as is_expensive_basket,
       CASE WHEN quantity = 1 THEN 1 ELSE 0 END as is_single_item,
       CASE WHEN quantity > 10 THEN 1 ELSE 0 END as is_bulk_item,
       CASE WHEN quantity < 0 AND invoiceno LIKE 'C%' THEN 1 ELSE 0 END AS is_return_line

       FROM mart.retail_enriched r
       """)

In [47]:
# Kollar stats på hur basket_size och basket_value ser ut. min,max,percentiler osv. Därefter väljs värden för features.
df_basket_stats = db.sql("""
SELECT distinct invoiceno, basket_value
FROM mart.retail_enriched
""").df()

df_basket_stats.describe()

,basket_value
count,25900.000000
mean,375.521504
std,2012.631335
min,-168469.600000
25%,11.775000
50%,206.305000
75%,407.385000
max,168469.600000


In [48]:
db.sql("SELECT * FROM mart.retail_enriched LIMIT 5")

┌────────────────────────────────────┬──────────┬─────────────────────┬───────────┬────────────┬────────────────┬───────────┬───────────┬───────────────────┬────────────┬───────────────┬────────────────────────┬──────────────────────┬─────────────────────┬──────────────────┬───────────────────────┬──────────────────────┬───────────────────┬───────────────────────┬──────────────────────┬──────────────────────┬────────────────────┬───────┬───────┬────────────────┬──────────────┬───────┬─────────────┬───────┬─────────┬──────────┬─────────┬────────────────┬──────────────────┬──────────────┬────────────────┬────────────┬───────────────────┬─────────────┬──────────────┬─────────────┬────────────────────┬────────────────────┬───────────────────────┬─────────────────┬─────────────────────┬────────────────┬──────────────┬────────────────┐
│            Description             │ Quantity │     InvoiceDate     │ UnitPrice │ CustomerID │    Country     │ stockcode │ invoiceno │ is_credit_invoice │ 

In [49]:
db.sql("""CREATE OR REPLACE TABLE mart.productdim AS
       SELECT r.stockcode,
       mode(r.description) as description,
       MAX(is_product) as is_product,

       ABS(SUM(CASE WHEN is_return_line = 1 THEN total_price END)) / SUM(CASE WHEN total_price > 0 THEN total_price END) as return_rate_value,
       ABS(SUM(CASE WHEN is_return_line = 1 THEN quantity END)) / SUM(CASE WHEN quantity > 0 THEN quantity END) as return_rate_quantity,
       COUNT(DISTINCT CASE WHEN is_return_line = 1 THEN invoiceno END) / COUNT(DISTINCT invoiceno) as return_rate_order,
       ABS(SUM(CASE WHEN is_return_line = 1 THEN quantity END)) as return_quantity,

       AVG(basket_size) as avg_basket_size,
       AVG(basket_value) as avg_basket_value,
       AVG(share_of_basket) as avg_share_of_basket,
       SUM(CASE WHEN quantity > 0 THEN quantity END) / NULLIF(COUNT(DISTINCT CASE WHEN quantity > 0 THEN invoiceno END), 0) as avg_units_sold,
       
       SUM(CASE WHEN quantity > 0 THEN quantity END) as total_quantity_sold,
       SUM(CASE WHEN total_price > 0 THEN total_price END) as total_sales,
       AVG(CASE WHEN quantity > 0 THEN unitprice END) AS avg_unit_price,

       COUNT(DISTINCT CASE WHEN quantity > 0 THEN country END) as unique_countries,
       COUNT(DISTINCT CASE WHEN quantity > 0 THEN customerid END) as unique_customers,

       COUNT(DISTINCT CASE WHEN quantity > 0 THEN invoiceno END) as orders_with_product,
       COUNT(DISTINCT CASE WHEN quantity > 0 THEN invoiceno END) / NULLIF((SELECT COUNT(DISTINCT invoiceno) FROM mart.retail_enriched WHERE quantity > 0), 0) as order_penetration,
       
       AVG(CASE WHEN quantity > 0 THEN  total_price END) as avg_line_revenue,
       COUNT(DISTINCT CASE WHEN quantity > 0 THEN invoiceno END) / NULLIF(COUNT(DISTINCT CASE WHEN quantity > 0 THEN customerid END), 0) as order_per_customer,

       MIN(CASE WHEN quantity > 0 THEN invoicedate END) as first_sale_date,
       MAX(CASE WHEN quantity > 0 THEN invoicedate END) as last_sale_date,
       MAX(CASE WHEN quantity > 0 THEN invoicedate END) - MIN(CASE WHEN quantity > 0 THEN invoicedate END) as product_lifetime

       FROM mart.retail_enriched r
       GROUP BY r.stockcode
       """)

In [50]:
# avg_days_between_orders
# Joinar senare in i customerdim
# Denna ch ett par features till skapas som separata tabeller på radnivå innan aggregering i customerdim.
db.sql("""
CREATE OR REPLACE TABLE customer_feature_avg_days_between AS

WITH order_dates AS (
SELECT DISTINCT
       customerid,
       invoiceno,
       invoicedate
FROM mart.retail_enriched
WHERE quantity > 0
),

order_gaps AS (
SELECT
       customerid,
       invoicedate,
       DATEDIFF(
            'day',
            LAG(invoicedate) OVER(
                PARTITION BY customerid
                ORDER BY invoicedate
            ),
            invoicedate
       ) AS days_between_orders
FROM order_dates
)

SELECT
       customerid,
       AVG(days_between_orders) AS avg_days_between_orders,
       STDDEV(days_between_orders) AS order_interval_variability
FROM order_gaps
WHERE customerid IS NOT NULL
GROUP BY customerid
""")

In [51]:
# favorite_purchase_hour
# Joinar senare in i customerdim
db.sql("""
CREATE OR REPLACE TABLE customer_feature_favorite_hour AS

WITH hour_counts AS (
SELECT
       customerid,
       hour,
       COUNT(*) AS purchases
FROM mart.retail_enriched
WHERE quantity > 0
GROUP BY customerid, hour
),

ranked_hours AS (
SELECT *,
       ROW_NUMBER() OVER(
            PARTITION BY customerid
            ORDER BY purchases DESC
       ) AS rn
FROM hour_counts
)

SELECT
       customerid,
       hour AS favorite_purchase_hour
FROM ranked_hours
WHERE rn = 1
""")

In [52]:
# favorite_purchase_day
# Även denna joinas senare in i customerdim
db.sql("""
CREATE OR REPLACE TABLE customer_feature_favorite_day AS

WITH day_counts AS (
SELECT
       customerid,
       weekday_name,
       COUNT(*) AS purchases
FROM mart.retail_enriched
WHERE quantity > 0
GROUP BY customerid, weekday_name
),

ranked_days AS (
SELECT *,
       ROW_NUMBER() OVER(
            PARTITION BY customerid
            ORDER BY purchases DESC
       ) AS rn
FROM day_counts
)

SELECT
       customerid,
       weekday_name AS favorite_purchase_day
FROM ranked_days
WHERE rn = 1
""")   


In [53]:
# top_product
# Joinas också in i customerdim
db.sql("""
CREATE OR REPLACE TABLE customer_feature_top_product AS

WITH product_counts AS (
SELECT
       customerid,
       stockcode,
       COUNT(*) AS purchases
FROM mart.retail_enriched
WHERE quantity > 0
AND is_product = 1
GROUP BY customerid, stockcode
),

ranked_products AS (
SELECT *,
       ROW_NUMBER() OVER(
            PARTITION BY customerid
            ORDER BY purchases DESC
       ) AS rn
FROM product_counts
)

SELECT
       customerid,
       stockcode AS top_product
FROM ranked_products
WHERE rn = 1
""")

In [54]:
# Skapar mart.customerdim - kunddimensionen
# Notering: total_spend i customerdim inkluderar endast rader där quantity > 0.
# För totala intäktsberäkningar, använd retail_enriched som källa.
# customerdim används för kundbeteendeanalys på identifierade kunder.
db.sql("""
CREATE OR REPLACE TABLE mart.customerdim AS
SELECT 
       a.customerid,

       d.avg_days_between_orders,
       d.order_interval_variability,
       h.favorite_purchase_hour,
       day.favorite_purchase_day,
       tp.top_product,

       COUNT(DISTINCT CASE WHEN quantity > 0 THEN invoiceno END) as total_orders,

       SUM(CASE WHEN total_price > 0 THEN total_price END) as total_spend,

       SUM(CASE WHEN total_price > 0 THEN total_price END) 
       / NULLIF(COUNT(DISTINCT CASE WHEN total_price > 0 THEN invoiceno END), 0)
       as avg_order_value,

       SUM(CASE WHEN quantity > 0 THEN quantity END) as total_units,

       SUM(CASE WHEN quantity > 0 THEN quantity END)
       / NULLIF(COUNT(DISTINCT CASE WHEN quantity > 0 THEN invoiceno END), 0)
       as avg_units_per_order,

       MAX(CASE WHEN quantity > 0 THEN invoicedate END) as last_purchase_date,

       DATEDIFF('day', 
            MAX(CASE WHEN quantity > 0 THEN invoicedate END),
            '2011-12-09'
       ) as days_since_last_purchase,

       COUNT(DISTINCT CASE WHEN quantity > 0 THEN invoiceno END)
       / NULLIF(
            DATEDIFF('month',
                MIN(CASE WHEN quantity > 0 THEN invoicedate END),
                MAX(CASE WHEN quantity > 0 THEN invoicedate END)
            ),
       0) as orders_per_month,

       MIN(CASE WHEN quantity > 0 THEN invoicedate END) as first_purchase_date,

       DATEDIFF('day',
            MIN(CASE WHEN quantity > 0 THEN invoicedate END),
            MAX(CASE WHEN quantity > 0 THEN invoicedate END)
       ) as customer_lifetime_days,

       COUNT(DISTINCT CASE WHEN quantity > 0 THEN stockcode END) as unique_items,

       SUM(CASE WHEN total_price > 0 THEN total_price END)
       / NULLIF(
            DATEDIFF('month',
                MIN(CASE WHEN quantity > 0 THEN invoicedate END),
                MAX(CASE WHEN quantity > 0 THEN invoicedate END)
            ),
       0) AS avg_monthly_spend,

       CASE
            WHEN COUNT(DISTINCT CASE WHEN quantity > 0 THEN invoiceno END) > 1
            THEN 1
            ELSE 0
       END AS is_repeat_buyer,

       SUM(CASE WHEN quantity < 0 THEN ABS(quantity) END) as return_units,

       COUNT(DISTINCT CASE WHEN quantity < 0 THEN invoiceno END)
       / NULLIF(COUNT(DISTINCT CASE WHEN quantity > 0 THEN invoiceno END),0)
       AS return_rate_order,

       SUM(CASE WHEN total_price < 0 THEN ABS(total_price) END)
       / NULLIF(SUM(CASE WHEN total_price > 0 THEN total_price END),0)
       AS return_rate_value

FROM mart.retail_enriched a

LEFT JOIN customer_feature_avg_days_between d
ON a.customerid = d.customerid

LEFT JOIN customer_feature_favorite_hour h
ON a.customerid = h.customerid

LEFT JOIN customer_feature_favorite_day day
ON a.customerid = day.customerid

LEFT JOIN customer_feature_top_product tp
ON a.customerid = tp.customerid
       
WHERE a.customerid IS NOT NULL
       
GROUP BY 
       a.customerid,
       d.avg_days_between_orders,
       d.order_interval_variability,
       h.favorite_purchase_hour,
       day.favorite_purchase_day,
       tp.top_product
""")

In [55]:
# Slutverifiering
print(db.sql("SELECT * FROM staging.retail_clean LIMIT 1").columns)
db.sql("""
    SELECT 
        COUNT(*) AS total_rows,
        COUNT(DISTINCT (invoiceno, stockcode, description, quantity, unitprice, invoicedate, customerid, country)) AS distinct_rows
    FROM staging.retail_clean
""")

['Description', 'Quantity', 'InvoiceDate', 'UnitPrice', 'CustomerID', 'Country', 'stockcode', 'invoiceno', 'is_credit_invoice', 'is_product', 'price_segment', 'is_missing_description', 'is_negative_quantity', 'is_extreme_quantity', 'is_zero_quantity', 'is_negative_unitprice', 'is_extreme_unitprice', 'is_zero_unitprice', 'is_missing_customerid', 'is_missing_stockcode', 'is_missing_invoiceno', 'is_unusual_country']


┌────────────┬───────────────┐
│ total_rows │ distinct_rows │
│   int64    │     int64     │
├────────────┼───────────────┤
│     536641 │        536641 │
└────────────┴───────────────┘

In [56]:
# För eventuella rapporter
tables = {
    'retail_enriched': 'mart.retail_enriched',
    'productdim': 'mart.productdim',
    'customerdim': 'mart.customerdim',
    'top_anomalies': 'staging.top_anomalies'
}

for name, table in tables.items():
    path = f'../Reports/{name}.parquet'
    db.execute(f"COPY (SELECT * FROM {table}) TO '{path}' (FORMAT PARQUET)")
    print(f"Exporterat: {name}")

Exporterat: retail_enriched
Exporterat: productdim
Exporterat: customerdim
Exporterat: top_anomalies
